# query

> The structured typed query expression (pass-2 Thread 5) — DATA nouns describing graph reads. **Execution lives in graph-storage adapters** (stage 4 translates expressions per-backend); this library only defines, validates, and (de)serializes them. Typed expressions are the primary, portable surface — no storage-schema leak (the raw-SQL `nodes`/`edges` + `json_extract` coupling of ledger C2/C3), and scale-shaped (server-side filter/page/count answering D13). `RawQuery` is the explicitly-marked, backend-coupled escape hatch: recurring raw patterns EXPOSE missing typed-expression capabilities and get PROMOTED into the typed surface (the real-world-testing forcing function).

In [ ]:
#| default_exp query

## Design notes

Replaces the old `GraphQuery` raw-string DTO. The expression power built now is what the cores' evidence demands — label + property filtering (C2/C3), batch-by-id (C17), relation traversal (corrections joins), source matching (`find_prior_corrections_by_hash`), order/page/count (D13 server-side aggregates) — while the shape admits a fuller query language later (generalize-seam-one-level).

**Operator vocabulary is reserved up front** (`PREDICATE_OPS`); executors implement progressively and MUST raise on unsupported operators, never silently ignore (reserve-enum-values-up-front discipline).

Deferred until adopter evidence: group-by aggregates, depth-N traversal expressions (whole-neighborhood reads stay on `get_context`), compound boolean predicates (AND is the implicit combination of `where`; OR/NOT wait for a real consumer).

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from cjm_plugin_system.core.errors import PluginInputError

from cjm_context_graph_primitives.locators import ResourceLocator, locator_from_dict

In [ ]:
#| export
PREDICATE_OPS = (  # Reserved operator vocabulary (executors may implement a subset but MUST raise on unsupported ops)
    "eq",        # property == value
    "ne",        # property != value
    "lt",        # property < value
    "le",        # property <= value
    "gt",        # property > value
    "ge",        # property >= value
    "in",        # property in value (list)
    "contains",  # value is a substring of property (text match)
    "is_null",   # property is null / absent (value must be None)
    "not_null",  # property is present and non-null (value must be None)
)

In [ ]:
#| export
@dataclass(frozen=True)
class PropertyPredicate:
    """One property comparison; a query's `where` list combines predicates with AND."""
    prop: str          # Property name inside the entity's properties payload
    op: str            # Operator from PREDICATE_OPS
    value: Any = None  # Comparison value (must be None for is_null/not_null)

    def __post_init__(self):
        if self.op not in PREDICATE_OPS:
            raise PluginInputError(
                f"Unknown predicate op {self.op!r} (known: {PREDICATE_OPS})",
                fields_invalid=["op"],
            )
        if self.op in ("is_null", "not_null") and self.value is not None:
            raise PluginInputError(
                f"Op {self.op!r} takes no value (got {self.value!r})",
                fields_invalid=["value"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {"prop": self.prop, "op": self.op, "value": self.value}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "PropertyPredicate":  # Reconstructed predicate
        """Reconstruct from the wire dict form."""
        return cls(prop=d["prop"], op=d["op"], value=d.get("value"))

In [ ]:
#| export
@dataclass(frozen=True)
class SourcePredicate:
    """Match nodes whose `sources` contain a reference matching by content hash
    and/or locator — the typed form of the reverse provenance index
    (`find_prior_corrections_by_hash`-style reads; identity-first per CR-19)."""
    content_hash: Optional[str] = None        # Match refs with this content hash (PRIMARY identity)
    locator: Optional[ResourceLocator] = None  # Match refs with this locator

    def __post_init__(self):
        if self.content_hash is None and self.locator is None:
            raise PluginInputError(
                "SourcePredicate requires content_hash and/or locator",
                fields_invalid=["content_hash", "locator"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {
            "content_hash": self.content_hash,
            "locator": self.locator.to_dict() if self.locator is not None else None,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "SourcePredicate":  # Reconstructed predicate
        """Reconstruct from the wire dict form."""
        loc = d.get("locator")
        return cls(
            content_hash=d.get("content_hash"),
            locator=locator_from_dict(loc) if loc is not None else None,
        )

In [ ]:
#| export
RELATION_DIRECTIONS = ("out", "in", "both")  # Edge direction relative to the candidate node


@dataclass(frozen=True)
class RelationPredicate:
    """Match nodes that have an edge of `relation_type` (one-hop, typed traversal —
    e.g. "Segments PART_OF document D"). Depth-N neighborhood reads stay on
    `get_context`; richer traversal expressions wait for adopter evidence."""
    relation_type: str             # Edge relation type, e.g. "PART_OF", "CORRECTS"
    direction: str = "out"         # Edge direction from the candidate node: out / in / both
    node_id: Optional[str] = None  # Constrain the far end to this node id (None = any)

    def __post_init__(self):
        if self.direction not in RELATION_DIRECTIONS:
            raise PluginInputError(
                f"Unknown direction {self.direction!r} (known: {RELATION_DIRECTIONS})",
                fields_invalid=["direction"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {"relation_type": self.relation_type, "direction": self.direction,
                "node_id": self.node_id}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "RelationPredicate":  # Reconstructed predicate
        """Reconstruct from the wire dict form."""
        return cls(relation_type=d["relation_type"], direction=d.get("direction", "out"),
                   node_id=d.get("node_id"))

In [ ]:
#| export
@dataclass(frozen=True)
class OrderBy:
    """Result ordering by one property (server-side; C2/C3 ORDER BY index)."""
    prop: str                 # Property name to order by
    descending: bool = False  # Descending order when True

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {"prop": self.prop, "descending": self.descending}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "OrderBy":  # Reconstructed ordering
        """Reconstruct from the wire dict form."""
        return cls(prop=d["prop"], descending=d.get("descending", False))

In [ ]:
#| export
@dataclass
class NodeQuery:
    """Typed node read: filter / traverse / order / page / project / count.

    All filter fields combine with AND. `count=True` returns a count instead of
    rows (the D13 verify-spine aggregate shape). `project` limits returned
    properties (server-side projection; None = whole nodes).
    """
    ids: Optional[List[str]] = None              # Batch-by-id (C17); None = no id filter
    label: Optional[str] = None                  # Node label filter
    where: List[PropertyPredicate] = field(default_factory=list)  # Property predicates (AND)
    source: Optional[SourcePredicate] = None     # Provenance reverse-index match
    related: Optional[RelationPredicate] = None  # One-hop relation constraint
    order_by: Optional[OrderBy] = None           # Server-side ordering
    limit: Optional[int] = None                  # Page size; None = backend default
    offset: int = 0                              # Page offset
    count: bool = False                          # Return count instead of rows
    project: Optional[List[str]] = None          # Property names to return; None = whole nodes

    TYPE = "node_query"  # Wire-dict type tag

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {
            "type": self.TYPE,
            "ids": self.ids,
            "label": self.label,
            "where": [p.to_dict() for p in self.where],
            "source": self.source.to_dict() if self.source is not None else None,
            "related": self.related.to_dict() if self.related is not None else None,
            "order_by": self.order_by.to_dict() if self.order_by is not None else None,
            "limit": self.limit,
            "offset": self.offset,
            "count": self.count,
            "project": self.project,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "NodeQuery":  # Reconstructed query
        """Reconstruct from the wire dict form."""
        src, rel, ob = d.get("source"), d.get("related"), d.get("order_by")
        return cls(
            ids=d.get("ids"),
            label=d.get("label"),
            where=[PropertyPredicate.from_dict(p) for p in d.get("where", [])],
            source=SourcePredicate.from_dict(src) if src is not None else None,
            related=RelationPredicate.from_dict(rel) if rel is not None else None,
            order_by=OrderBy.from_dict(ob) if ob is not None else None,
            limit=d.get("limit"),
            offset=d.get("offset", 0),
            count=d.get("count", False),
            project=d.get("project"),
        )

In [ ]:
#| export
@dataclass
class EdgeQuery:
    """Typed edge read: filter / order / page / project / count.

    Covers the cores' edge reads: counting edges by relation type for a spine
    (D13 verify aggregates) and reading edge properties off a node's edges
    (correction decisions).
    """
    ids: Optional[List[str]] = None       # Batch-by-id; None = no id filter
    relation_type: Optional[str] = None   # Edge relation-type filter
    source_id: Optional[str] = None       # Origin node filter
    target_id: Optional[str] = None       # Destination node filter
    where: List[PropertyPredicate] = field(default_factory=list)  # Property predicates (AND)
    order_by: Optional[OrderBy] = None    # Server-side ordering
    limit: Optional[int] = None           # Page size; None = backend default
    offset: int = 0                       # Page offset
    count: bool = False                   # Return count instead of rows
    project: Optional[List[str]] = None   # Property names to return; None = whole edges

    TYPE = "edge_query"  # Wire-dict type tag

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {
            "type": self.TYPE,
            "ids": self.ids,
            "relation_type": self.relation_type,
            "source_id": self.source_id,
            "target_id": self.target_id,
            "where": [p.to_dict() for p in self.where],
            "order_by": self.order_by.to_dict() if self.order_by is not None else None,
            "limit": self.limit,
            "offset": self.offset,
            "count": self.count,
            "project": self.project,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "EdgeQuery":  # Reconstructed query
        """Reconstruct from the wire dict form."""
        ob = d.get("order_by")
        return cls(
            ids=d.get("ids"),
            relation_type=d.get("relation_type"),
            source_id=d.get("source_id"),
            target_id=d.get("target_id"),
            where=[PropertyPredicate.from_dict(p) for p in d.get("where", [])],
            order_by=OrderBy.from_dict(ob) if ob is not None else None,
            limit=d.get("limit"),
            offset=d.get("offset", 0),
            count=d.get("count", False),
            project=d.get("project"),
        )

In [ ]:
#| export
@dataclass
class RawQuery:
    """The explicitly-marked, backend-coupled escape hatch.

    For reads the typed expressions cannot yet express. `backend` is REQUIRED —
    a raw query is non-portable by construction and an executor must refuse a
    backend mismatch. Recurring raw patterns are promotion candidates into the
    typed surface (record them; that is the forcing function).
    """
    text: str      # Backend-native query text (e.g. SQL)
    backend: str   # Backend this query is coupled to (e.g. "sqlite"); executors refuse mismatches
    params: Any = field(default_factory=list)  # Positional list or named dict of parameters

    TYPE = "raw_query"  # Wire-dict type tag

    def __post_init__(self):
        if not self.backend:
            raise PluginInputError(
                "RawQuery requires an explicit backend (it is non-portable by construction)",
                fields_invalid=["backend"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {"type": self.TYPE, "text": self.text, "backend": self.backend,
                "params": self.params}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "RawQuery":  # Reconstructed query
        """Reconstruct from the wire dict form."""
        return cls(text=d["text"], backend=d["backend"], params=d.get("params", []))

In [ ]:
#| export
QUERY_TYPES: Dict[str, type] = {  # Tagged-dict registry for wire dispatch
    NodeQuery.TYPE: NodeQuery,
    EdgeQuery.TYPE: EdgeQuery,
    RawQuery.TYPE: RawQuery,
}


def query_from_dict(
    d: Dict[str, Any]  # Tagged wire dict ("type" in QUERY_TYPES)
) -> Any:  # NodeQuery | EdgeQuery | RawQuery
    """Reconstruct a query expression from its tagged wire dict."""
    t = d.get("type")
    cls = QUERY_TYPES.get(t)
    if cls is None:
        raise PluginInputError(
            f"Unknown query type {t!r} (known: {sorted(QUERY_TYPES)})",
            fields_invalid=["type"],
        )
    return cls.from_dict(d)

In [ ]:
# The three evidenced core read patterns, expressed typed (documentation-as-test):

# 1. C2/C3 spine read: Segments PART_OF a document, ordered by index, projected
spine = NodeQuery(
    label="Segment",
    related=RelationPredicate(relation_type="PART_OF", direction="out", node_id="doc-1"),
    order_by=OrderBy(prop="index"),
    project=["index", "text", "start_time", "end_time"],
)
assert query_from_dict(spine.to_dict()) == spine

# 2. Prune-candidate read: empty-or-null text Segments (is_null half; eq half)
empties = NodeQuery(
    label="Segment",
    where=[PropertyPredicate(prop="text", op="eq", value="")],
    related=RelationPredicate(relation_type="PART_OF", node_id="doc-1"),
    count=True,
)
assert query_from_dict(empties.to_dict()) == empties

# 3. Cross-transcript correction-cache read: Corrections whose CORRECTS target
#    carried this content hash (SourcePredicate would sit on the traversal far end;
#    built here as the two-step the adapter can compose)
by_hash = NodeQuery(source=SourcePredicate(content_hash="sha256:ab"))
assert query_from_dict(by_hash.to_dict()) == by_hash

# D13 verify aggregate: count NEXT edges (server-side; no neighborhood materialization)
next_count = EdgeQuery(relation_type="NEXT", count=True)
assert query_from_dict(next_count.to_dict()) == next_count

# batch-by-id (C17)
batch = NodeQuery(ids=["n1", "n2", "n3"])
assert query_from_dict(batch.to_dict()) == batch

# raw escape is marked: backend required
raw = RawQuery(text="SELECT COUNT(*) FROM nodes", backend="sqlite")
assert query_from_dict(raw.to_dict()) == raw
try:
    RawQuery(text="SELECT 1", backend="")
    raise AssertionError("should have raised")
except PluginInputError:
    pass

# validation: unknown op / bad direction / valueless null-op
from cjm_plugin_system.core.errors import PluginInputError as PIE
for bad in (lambda: PropertyPredicate("text", "like", "%x%"),
            lambda: PropertyPredicate("text", "is_null", value=1),
            lambda: RelationPredicate("NEXT", direction="up"),
            lambda: SourcePredicate()):
    try:
        bad()
        raise AssertionError("should have raised")
    except PIE:
        pass

# unknown query type raises
try:
    query_from_dict({"type": "graph_query"})
    raise AssertionError("should have raised")
except PIE:
    pass